# Comparação de modelos, versão 2

Esta é a segunda versão do caderno de comparação. A primeira permanece intacta em `comparacao_modelos.ipynb` e continua sendo o registro da comparação original entre floresta aleatória, gradiente e a mescla dos dois.

Três coisas mudam aqui.

O Extra Trees passa a ser comparado junto com os demais, sob o mesmo protocolo e o mesmo espaço de busca. Ele entrou porque o melhor resultado de Main Vent Sys, na formulação entregue do caderno de modelagem, é a média entre o Extra Trees e o gradiente, e passa a ser medido também aqui, isoladamente, ao lado dos demais.

O quadro consolidado da Seção 11 reúne as sete áreas com o melhor resultado de cada uma entre os treinos feitos nos cadernos de `colabs_S4`, o erro médio absoluto e o coeficiente de determinação, e registra em colunas de sim ou não qual tratamento das perdas e qual tratamento do motivo do ensaio aquele resultado utiliza.

Quatro áreas trazem o melhor resultado de outro caderno de `colabs_S4`, e não deste. Oven HMI, Main Vent Sys e IPC Comp vêm da formulação entregue de `modelagem.ipynb`, e IPC Cooling vem de `random_search_13cat.ipynb`. Os quatro casos estão declarados como constantes na Seção 11, com a origem e o tratamento de dados de cada um, porque foram medidos com outra receita ou outra preparação da base.

A Seção 7.1 mostra, com o Oven HMI como exemplo, quanto as métricas melhorariam se a validação fosse feita por linha em vez de por configuração, e por que esse corte não foi adotado.

A busca aleatória segue o método do caderno `random_search_13cat`, desenvolvido por outra frente do grupo, reexecutado aqui sobre a base corrigida, que passou a incluir as colunas do motivo do ensaio.

## 1. Como executar

No Google Colab, a primeira célula de código pede o envio dos arquivos de dados, que não estão no repositório por compromisso de proteção de dados com a parceira. São nove arquivos: os sete conjuntos por área, com nome dataset seguido da área e do sufixo 13cat, a tabela `limites_ativos.csv` e a partição `particao_treino_teste.json`.

Fora do Colab, o caderno procura os arquivos na própria pasta e nas pastas do repositório, e dispensa o envio.

A versão anterior levou cerca de onze minutos, quase todos na busca de hiperparâmetros. Esta versão busca um algoritmo a mais, de modo que o tempo esperado sobe para algo em torno de dezesseis minutos. No Colab gratuito, que dispõe de menos núcleos de processamento, o tempo tende a ser maior.

As quatro áreas importadas de outros cadernos não são recalculadas aqui: os seus valores estão escritos como constantes na Seção 11.

In [ ]:
# == caminhos do projeto ==
# A pasta deste caderno, dentro de notebooks/. Os CSVs sao lidos e gravados
# nela; as leituras entre pastas usam ../<outra pasta>. Para que esses caminhos
# valham qualquer que seja o diretorio em que o kernel foi aberto, o bloco
# abaixo localiza a raiz do repositorio -- a pasta que contem notebooks/ e
# documents/ -- e entra na pasta do caderno. No Colab, onde nao existe
# repositorio e os arquivos ficam soltos ao lado do caderno, nada e alterado.
import os

PASTA_CADERNO = "colabs_S4"


def raiz_repositorio(inicio=None):
    """Sobe a arvore de diretorios ate a raiz do repositorio; None se nao achar."""
    atual = os.path.abspath(inicio or os.getcwd())
    while True:
        if all(os.path.isdir(os.path.join(atual, p)) for p in ("notebooks", "documents")):
            return atual
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai


RAIZ_REPO = raiz_repositorio()
if RAIZ_REPO:
    os.chdir(os.path.join(RAIZ_REPO, "notebooks", PASTA_CADERNO))
print("Diretorio de trabalho:", os.path.abspath("."))

In [ ]:
import json
import os
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import loguniform, randint, uniform
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import (ExtraTreesRegressor, GradientBoostingRegressor,
                              RandomForestRegressor)
from sklearn.model_selection import KFold, LeaveOneGroupOut, RandomizedSearchCV
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")

SEMENTE = 42
AREAS = ["oven_hmi", "ct_knobs", "main_vent_sys", "ipc_cooling", "mainback", "ipc_comp", "coil_temps"]
ARQUIVOS = [f"dataset_{a}_13cat.csv" for a in AREAS] + ["limites_ativos.csv", "particao_treino_teste.json"]

try:
    from google.colab import files
    faltando = [f for f in ARQUIVOS if not os.path.exists(f)]
    if faltando:
        print("Envie os arquivos:", faltando)
        files.upload()
except ImportError:
    pass


def localiza(nome):
    """Procura o arquivo na pasta atual e nas pastas do repositorio.

    Os CSVs sao lidos apenas do colabs_S4; so a particao JSON pode vir da pasta de preparacao."""
    pastas = (".", "notebooks/colabs_S4")
    if not nome.endswith(".csv"):
        pastas += ("../colabs_S3/preparacao_testes", "notebooks/colabs_S3/preparacao_testes")
    for pasta in pastas:
        caminho = os.path.join(pasta, nome)
        if os.path.exists(caminho):
            return caminho
    raise FileNotFoundError(f"{nome} nao encontrado. Envie o arquivo ou execute a partir do repositorio.")


print("Arquivos localizados:", all(os.path.exists(localiza(f)) for f in ARQUIVOS))

## 2. Os dados e o protocolo de validação

Cada área térmica tem um conjunto próprio, com uma linha por ensaio e uma coluna por componente. O conjunto é convertido para o formato longo, com uma linha por par de ensaio e componente, de modo que um único modelo por área aprenda todos os componentes daquela área.

As variáveis de entrada são os atributos de configuração do ensaio, o perfil de perdas imposto às bobinas e o motivo do ensaio, este codificado em seis categorias mutuamente exclusivas. As colunas de ordem de acionamento das bobinas ficam de fora, por terem sido medidas como prejudiciais no caderno de modelagem.

A validação retira uma configuração experimental inteira por vez, com todos os seus componentes e todos os ensaios que a repetem, treina com as demais e prevê a que ficou de fora. São dezessete dobras. Uma divisão aleatória das linhas colocaria componentes do mesmo ensaio nos dois lados, e o modelo passaria a reconhecer a configuração em vez de generalizar para uma nova; medido nesta base, esse atalho faz o erro parecer quarenta e cinco por cento menor. A Seção 7.1 mostra o mesmo efeito no Oven HMI.

Os dez ensaios do conjunto de teste reservado não são lidos neste caderno.

In [ ]:
ENTRADAS = ("blower_direction_", "housing_", "board_position_", "run_case_", "losses_",
            "deflector_hole_", "material_coil_support_", "reason_")
EXCLUIDAS = ("loss_profile_order_",)

particao = json.load(open(localiza("particao_treino_teste.json"), encoding="utf-8"))
DOBRA = {int(k): v for k, v in particao["dobra_por_run_idx"].items()}
TESTE = set(particao["run_idx_teste"])
LIMITES = pd.read_csv(localiza("limites_ativos.csv")).set_index("tc_name")["limite_num"]


def carrega(area):
    """Formato longo da area, apenas com os ensaios de desenvolvimento."""
    d = pd.read_csv(localiza(f"dataset_{area}_13cat.csv"))
    d["run_idx"] = np.arange(len(d))
    entradas = [c for c in d.columns if c.startswith(ENTRADAS)]
    alvos = [c for c in d.columns if c in LIMITES.index]
    sobra = [c for c in d.columns if c not in entradas + alvos + ["run_idx"] and not c.startswith(EXCLUIDAS)]
    assert not sobra, f"{area}: colunas nao classificadas: {sobra}"
    longo = (d[d["run_idx"].map(lambda r: DOBRA.get(r) is not None and r not in TESTE)]
             .melt(id_vars=entradas + ["run_idx"], value_vars=alvos,
                   var_name="componente", value_name="temperatura")
             .dropna(subset=["temperatura"])
             .reset_index(drop=True))
    X = longo[entradas + ["componente"]]
    y = longo["temperatura"].to_numpy()
    grupos = longo["run_idx"].map(DOBRA).astype(int).to_numpy()
    return X, y, grupos


DADOS = {a: carrega(a) for a in AREAS}
for a, (X, y, g) in DADOS.items():
    print(f"{a:<15} {len(y):>5} leituras | {X['componente'].nunique():>2} componentes | "
          f"{len(set(g))} dobras | {X.shape[1] - 1} variaveis de entrada")
print("Total:", sum(len(v[1]) for v in DADOS.values()), "leituras")

## 3. As métricas, e por que estas

O uso pretendido do modelo é uma triagem prévia: antecipar quais componentes provavelmente ultrapassarão o seu limite de segurança, para que o ensaio físico seja dirigido a eles. As métricas foram escolhidas a partir desse uso.

O erro médio absoluto, em graus Celsius, é a métrica principal. É a única expressa na unidade do problema e diretamente comparável à folga de projeto de cada componente, e é ela que decide a escolha entre modelos neste caderno.

Três métricas medem a triagem, sob a margem de segurança de oito graus acordada com a parceira, isto é, um componente é sinalizado quando a previsão somada a oito graus atinge o seu limite. As aprovações indevidas contam os componentes que ultrapassaram o limite e que a triagem deixou passar; é o erro caro, porque deixa passar um risco. As reprovações desnecessárias contam os componentes sinalizados sem necessidade; é o erro barato, porque custa um reensaio. A detecção é a fração dos estouros efetivamente sinalizada.

A raiz do erro quadrático médio acompanha o erro médio, por penalizar mais os erros grandes. O viés, que é a média dos erros com sinal, indica se o modelo tende a subestimar, o que é o lado perigoso para uma triagem.

O coeficiente de determinação consta das tabelas, mas não decide nada: ele compara o erro à variação total dos dados, de modo que uma área de temperaturas pouco variáveis tem coeficiente baixo mesmo com previsões boas.

In [ ]:
MARGEM = 8.0


def metricas(y, pred, componentes):
    erro = pred - y
    lim = componentes.map(LIMITES).to_numpy(dtype=float)
    com_limite = ~np.isnan(lim)
    estoura = y[com_limite] >= lim[com_limite]
    alarme = pred[com_limite] + MARGEM >= lim[com_limite]
    return {"MAE": np.abs(erro).mean(),
            "RMSE": np.sqrt((erro ** 2).mean()),
            "R2": 1 - (erro ** 2).sum() / ((y - y.mean()) ** 2).sum(),
            "vies": erro.mean(),
            "aprov_indevidas": int((estoura & ~alarme).sum()),
            "reprov_desnecessarias": int((~estoura & alarme).sum()),
            "deteccao_%": 100 * (estoura & alarme).sum() / estoura.sum() if estoura.sum() else np.nan,
            "estouros": int(estoura.sum())}

## 4. Os modelos

A floresta aleatória constrói centenas de árvores de decisão independentes, cada uma treinada sobre uma amostra diferente dos dados e considerando um subconjunto das variáveis a cada divisão, e devolve a média das previsões. Os erros individuais das árvores, sendo em boa parte independentes, se cancelam na média (BREIMAN, 2001).

O gradiente constrói árvores rasas em sequência, cada uma treinada para corrigir o erro deixado pelas anteriores. Onde a floresta reduz variância pela média, o gradiente reduz viés pelo encadeamento (FRIEDMAN, 2001).

O Extra Trees é uma floresta que sorteia o ponto de corte de cada divisão em vez de procurar o melhor. O sorteio acrescenta aleatoriedade e, com isso, reduz ainda mais a variância, ao custo de um viés um pouco maior em cada árvore (GEURTS; ERNST; WEHENKEL, 2006). Ele é o quarto algoritmo desta versão e não estava na primeira.

A mescla é a média aritmética das previsões da floresta e do gradiente. Como eles erram de formas diferentes, a média aproveita a complementaridade. Ela foi mantida como na primeira versão, com dois componentes e não três; a mescla de três algoritmos foi medida à parte, em `teste_mesclagem_extratrees.ipynb`.

Nos quatro casos, a identidade do componente é substituída por duas estatísticas calculadas a partir das temperaturas dele: o patamar, que é a média, e a escala, que é o desvio-padrão. Essas estatísticas são derivadas do próprio alvo, e por isso um transformador as calcula dentro do pipeline, usando apenas as linhas de treino de cada dobra. O vazamento de informação do alvo fica impedido por construção. Este transformador foi desenvolvido no caderno de busca aleatória e é reaproveitado aqui.

In [ ]:
class EstatisticasComponente(BaseEstimator, TransformerMixin):
    """Substitui a identidade do componente por patamar e escala, ajustados so no treino."""

    def fit(self, X, y):
        y = pd.Series(np.asarray(y, dtype=float), index=X.index)
        grupo = y.groupby(X["componente"])
        # Mesma regra do caderno de modelagem: escala 1 quando o desvio nao pode ser calculado
        # (um unico valor no treino) ou quando e zero.
        self.patamar_, self.escala_ = grupo.mean(), grupo.std().fillna(1.0).replace(0, 1.0)
        self.patamar_geral_, self.escala_geral_ = float(y.mean()), float(y.std())
        return self

    def transform(self, X):
        comp = X["componente"]
        saida = X.drop(columns="componente").copy()
        saida["patamar"] = comp.map(self.patamar_).fillna(self.patamar_geral_).to_numpy()
        saida["escala"] = comp.map(self.escala_).fillna(self.escala_geral_).to_numpy()
        return saida


def pipeline(estimador):
    return Pipeline([("componente", EstatisticasComponente()), ("modelo", estimador)])


def floresta(**p):
    return pipeline(RandomForestRegressor(random_state=SEMENTE, n_jobs=1, **p))


def gradiente(**p):
    return pipeline(GradientBoostingRegressor(random_state=SEMENTE, **p))


def extra(**p):
    return pipeline(ExtraTreesRegressor(random_state=SEMENTE, n_jobs=1, **p))


def divisoes(area, X, y, grupos):
    """Retira uma configuracao experimental inteira por vez, em todas as areas.

    A Secao 7.1 mostra, no Oven HMI, o que acontece com as metricas quando o corte e
    feito por linha, e por que esse corte nao foi adotado.
    """
    return list(LeaveOneGroupOut().split(X, y, grupos))


def prever(fabrica, X, y, divs):
    """Previsao de cada leitura quando ela esta fora do treino da sua divisao."""
    pred = np.empty(len(y))
    for treino, teste in divs:
        modelo = fabrica().fit(X.iloc[treino], y[treino])
        pred[teste] = modelo.predict(X.iloc[teste])
    return pred

## 5. Configuração fixa contra busca de hiperparâmetros

Hiperparâmetros são os valores de configuração do próprio algoritmo, tais como o número de árvores, a profundidade máxima e o número mínimo de exemplos por folha.

Cada algoritmo é avaliado em duas configurações. A configuração fixa é a do caderno de modelagem: trezentas árvores e todos os demais valores no padrão da biblioteca. A configuração buscada é a melhor entre quinze sorteadas por busca aleatória (BERGSTRA; BENGIO, 2012), com o mesmo espaço de busca do caderno de busca aleatória. O número de candidatos foi reduzido de quarenta para quinze para que a execução caiba numa sessão do Colab; com a mesma semente, os primeiros candidatos sorteados são os mesmos.

O Extra Trees percorre o mesmo espaço da floresta, sem a subamostragem de linhas, que não se aplica a ele porque não usa reamostragem por padrão. É o mesmo espaço que o caderno de busca aleatória lhe atribui.

A busca usa a mesma validação agrupada por configuração, e escolhe pelo erro médio absoluto. Para cada algoritmo e área, é mantida a melhor das duas configurações.

A mescla não é buscada separadamente: ela combina, em cada área, a floresta e o gradiente que venceram as suas respectivas comparações.

In [ ]:
N_CANDIDATOS = 15
ESPACO_ARVORE = {
    "modelo__n_estimators": randint(100, 601),
    "modelo__max_depth": [None, 4, 6, 8, 12, 16],
    "modelo__min_samples_leaf": randint(1, 11),
    "modelo__min_samples_split": randint(2, 11),
    "modelo__max_features": [1.0, 0.8, 0.6, 0.5, 0.33],
}
ESPACOS = {
    "floresta": {**ESPACO_ARVORE, "modelo__max_samples": [None, 0.6, 0.8]},
    # O Extra Trees nao reamostra linhas por padrao, de modo que max_samples nao se
    # aplica a ele. E o mesmo espaco que o caderno de busca aleatoria lhe atribui.
    "extra": dict(ESPACO_ARVORE),
    "gradiente": {
        "modelo__n_estimators": randint(100, 801),
        "modelo__learning_rate": loguniform(0.01, 0.2),
        "modelo__max_depth": randint(2, 7),
        "modelo__min_samples_leaf": randint(1, 16),
        "modelo__subsample": uniform(0.5, 0.5),
        "modelo__max_features": [None, 0.8, 0.6],
        "modelo__loss": ["squared_error", "huber", "absolute_error"],
    },
}
BASES = {"floresta": floresta, "gradiente": gradiente, "extra": extra}
FIXAS = {alg: (lambda f=f: f(n_estimators=300)) for alg, f in BASES.items()}
PUROS = ("floresta", "gradiente", "extra")

RESULTADO, ESCOLHA, PREVISAO = [], {}, {}
inicio = time.time()
for area in AREAS:
    X, y, g = DADOS[area]
    divs = divisoes(area, X, y, g)
    for alg in PUROS:
        pred_fixa = prever(FIXAS[alg], X, y, divs)
        busca = RandomizedSearchCV(BASES[alg](), ESPACOS[alg], n_iter=N_CANDIDATOS,
                                   scoring="neg_mean_absolute_error", cv=divs,
                                   n_jobs=-1, random_state=SEMENTE, refit=False)
        busca.fit(X, y)
        melhores = {k.replace("modelo__", ""): v for k, v in busca.best_params_.items()}
        pred_busca = prever(lambda: BASES[alg](**melhores), X, y, divs)
        mae_fixa, mae_busca = np.abs(pred_fixa - y).mean(), np.abs(pred_busca - y).mean()
        venceu = "busca" if mae_busca < mae_fixa else "fixa"
        ESCOLHA[(area, alg)] = melhores if venceu == "busca" else {"n_estimators": 300}
        PREVISAO[(area, alg)] = pred_busca if venceu == "busca" else pred_fixa
        RESULTADO.append({"area": area, "algoritmo": alg, "MAE_fixa": mae_fixa,
                          "MAE_busca": mae_busca, "mantida": venceu})
    # A mescla continua sendo a media de floresta e gradiente, como na versao 1.
    PREVISAO[(area, "mescla")] = (PREVISAO[(area, "floresta")] + PREVISAO[(area, "gradiente")]) / 2
    print(f"{area:<15} concluida | {(time.time() - inicio) / 60:.1f} min", flush=True)

TAB_BUSCA = pd.DataFrame(RESULTADO)
TAB_BUSCA["ganho"] = TAB_BUSCA["MAE_fixa"] - TAB_BUSCA["MAE_busca"]
print()
print(TAB_BUSCA.round(3).to_string(index=False))

### Hiperparâmetros mantidos por área

In [ ]:
def legivel(v):
    return round(float(v), 4) if isinstance(v, (float, np.floating)) else v


for (area, alg), p in ESCOLHA.items():
    print(f"{area:<15} {alg:<10}", {k: legivel(v) for k, v in p.items()})

## 6. Resultado por área

A célula abaixo traz o erro médio absoluto e o coeficiente de determinação de cada algoritmo em cada área, e em seguida isola o melhor algoritmo de cada área com as duas métricas dele. A última tabela impressa repete essa seleção em markdown, para que a Seção 7 possa ser atualizada sem transcrever número a mão.

In [ ]:
NOMES = {"floresta": "Floresta", "gradiente": "Gradiente",
         "extra": "Extra Trees", "mescla": "Mescla RF + GB"}
ROTULO = {"oven_hmi": "Oven HMI", "ct_knobs": "CT Knobs", "main_vent_sys": "Main Vent Sys",
          "ipc_cooling": "IPC Cooling", "mainback": "Mainback", "ipc_comp": "IPC Comp",
          "coil_temps": "Coil Temps"}
linhas = []
for area in AREAS:
    X, y, g = DADOS[area]
    for alg in NOMES:
        linhas.append({"area": area, "modelo": NOMES[alg],
                       **metricas(y, PREVISAO[(area, alg)], X["componente"])})
TAB_AREA = pd.DataFrame(linhas)

MODELOS = list(NOMES.values())
mae = TAB_AREA.pivot(index="area", columns="modelo", values="MAE").reindex(AREAS)[MODELOS]
r2 = TAB_AREA.pivot(index="area", columns="modelo", values="R2").reindex(AREAS)[MODELOS]
mae["vencedor"] = mae.idxmin(axis=1)

print("Erro medio absoluto por area e modelo, em graus Celsius:")
print(mae.round(3).to_string())
print()
print("Coeficiente de determinacao por area e modelo:")
print(r2.round(3).to_string())

# O vencedor de cada area e as duas metricas do proprio vencedor. O coeficiente de
# determinacao acompanha o erro, mas nao decide: ele e baixo onde a area varia pouco,
# ainda que a previsao esteja boa, conforme a Secao 3.
VENCEDOR = pd.DataFrame([
    {"area": ROTULO[a],
     "melhor_algoritmo": mae.loc[a, "vencedor"],
     "MAE": mae.loc[a, mae.loc[a, "vencedor"]],
     "R2": r2.loc[a, mae.loc[a, "vencedor"]],
     "segundo": mae.loc[a, MODELOS].drop(mae.loc[a, "vencedor"]).idxmin(),
     "MAE_segundo": mae.loc[a, MODELOS].drop(mae.loc[a, "vencedor"]).min()}
    for a in AREAS])
print()
print("Melhor algoritmo de cada area neste caderno, com as metricas dele:")
print(VENCEDOR.round(3).to_string(index=False))

In [ ]:
def agrega(modelo_por_area):
    """Metricas das sete areas juntas, com um modelo escolhido para cada uma."""
    y = np.concatenate([DADOS[a][1] for a in AREAS])
    pred = np.concatenate([PREVISAO[(a, modelo_por_area[a])] for a in AREAS])
    comp = pd.concat([DADOS[a][0]["componente"] for a in AREAS], ignore_index=True)
    return metricas(y, pred, comp)


INVERSO = {v: k for k, v in NOMES.items()}
agregado = {NOMES[alg]: agrega({a: alg for a in AREAS}) for alg in NOMES}
agregado["Melhor por área"] = agrega({a: INVERSO[mae.loc[a, "vencedor"]] for a in AREAS})
TAB_AGREGADO = pd.DataFrame(agregado).T
print(TAB_AGREGADO.round(3).to_string())

## 7. O que foi escolhido, e por quê

A busca de hiperparâmetros teve efeitos opostos na floresta e no gradiente, e o sinal foi o mesmo nas sete áreas.

Na floresta aleatória, a configuração fixa venceu a buscada em todas as áreas, com diferenças entre 0,059 e 1,137 grau. A floresta com os valores padrão da biblioteca já trabalha com árvores profundas e sem restrição, cujos erros se cancelam na média de trezentas árvores. O espaço de busca só oferece formas de restringi-la, limitando a profundidade, exigindo mais exemplos por folha ou considerando menos variáveis por divisão, e restringir piorou. A varredura do caderno de modelagem havia chegado à mesma conclusão por outro caminho.

No gradiente, a configuração buscada venceu a fixa em todas as áreas, com ganhos entre 0,034 e 1,470 grau. O gradiente é sensível à taxa de aprendizado e à profundidade das árvores, e os valores padrão, taxa de um décimo e profundidade três, não são os melhores para estes dados. As configurações vencedoras usam subamostragem das linhas em todas as áreas, profundidade entre três e cinco e número de árvores entre 147 e 661. Nenhuma delas combina a taxa de um décimo com a profundidade três, que são justamente os dois valores padrão. A mesma semente sorteia os mesmos quinze candidatos em todas as áreas, e por isso algumas áreas elegem a mesma configuração.

Esses dois parágrafos descrevem medições que não mudaram entre as duas versões: o Extra Trees acrescenta uma coluna à comparação e não altera os números da floresta, do gradiente nem da mescla dos dois. No agregado das sete áreas, a floresta erra 6,474 graus, o gradiente 5,954 e a mescla 6,075, como na primeira versão. Com o Extra Trees incluído, o agregado da escolha do melhor modelo por área deste caderno ficou em 5,950 graus, o mesmo da primeira versão.

Com cada algoritmo na sua melhor configuração, o melhor algoritmo de cada área, entre os treinados neste caderno, é o da última tabela da Seção 6, que traz também o erro médio absoluto e o coeficiente de determinação dele. A tabela é impressa pela célula, e não transcrita aqui, para que não passe a divergir dos números em caso de nova execução. O resultado final de cada área, no quadro da Seção 11, considera também os treinos feitos nos outros cadernos de `colabs_S4`.

Uma ressalva acompanha esses números e não deve ser separada deles. O erro de qualquer configuração escolhida por busca é otimista, porque a configuração foi escolhida pela mesma validação que depois a avalia. O tamanho desse otimismo não foi medido para cada algoritmo; no caderno de modelagem, uma busca composta sobretudo por florestas mostrou que noventa por cento do ganho aparente era o próprio ato de escolher. Aqui, porém, há um indício de que parte do ganho do gradiente é real: ele aparece nas sete áreas, com o mesmo sinal, e é grande onde a área é mais quente, chegando a 1,470 grau em Coil Temps. Um ganho produzido apenas pela escolha tenderia a ser pequeno e errático entre áreas.

### 7.1 Por que a validação é por configuração: o exemplo do Oven HMI

A tabela abaixo avalia o mesmo modelo, com a mesma base e os mesmos hiperparâmetros, sob dois cortes. No corte por configuração, adotado, a configuração prevista nunca aparece no treino, que é a situação real de uso: prever um fogão ainda não ensaiado. No corte KFold de cinco partes por linha, as outras leituras do mesmo ensaio ficam no treino, e o modelo passa a reconhecer o ensaio em vez de generalizar para um novo.

O corte por linha faz as métricas parecerem muito melhores, mas esse ganho não existe no uso real. Por isso o projeto adota o corte por configuração em todas as áreas. O modelo usado aqui é o gradiente deste caderno; o melhor resultado do Oven HMI, que entra no quadro da Seção 11, vem do caderno de modelagem, com outra receita, e por isso os números não coincidem.

O coeficiente de determinação baixo do Oven HMI no corte adotado vem da própria área, e não do modelo: as temperaturas variam pouco, com desvio-padrão de 3,9 graus contra 45,7 em Coil Temps, e os três componentes ficam em patamares parecidos. Com tão pouca variação a explicar, um erro de pouco mais de dois graus já deixa o coeficiente perto de zero.

In [ ]:
# Mesmo modelo, mesma base e mesmos hiperparametros: so o corte muda.
X_o, y_o, g_o = DADOS["oven_hmi"]
fabrica_o = lambda: gradiente(**ESCOLHA[("oven_hmi", "gradiente")])
CORTES = {
    "Por configuração (adotado)": list(LeaveOneGroupOut().split(X_o, y_o, g_o)),
    "KFold 5 por linha (não adotado)": list(KFold(5, shuffle=True, random_state=SEMENTE).split(X_o, y_o)),
}
linhas_corte = []
for nome, divs in CORTES.items():
    m = metricas(y_o, prever(fabrica_o, X_o, y_o, divs), X_o["componente"])
    linhas_corte.append({"validação": nome, "MAE": m["MAE"], "R2": m["R2"]})
print("Oven HMI, gradiente com os mesmos hiperparâmetros nos dois cortes:")
print(pd.DataFrame(linhas_corte).round(3).to_string(index=False))

## 8. Explicabilidade

A explicabilidade é apresentada para a floresta aleatória em IPC Comp, a área que concentra metade das leituras do projeto. A floresta é explicada porque é o estimador do melhor resultado de IPC Comp no quadro da Seção 11, que vem da formulação entregue do caderno de modelagem. Ela é ajustada aqui com as perdas nas treze categorias detalhadas e sem o peso junto ao limite, ao passo que a entrega usa as perdas agrupadas e o peso naquela área, de modo que a participação atribuída ao perfil de perdas descreve a versão detalhada e não a entregue. O método é o de valores de Shapley aditivos (LUNDBERG; LEE, 2017), que atribui a cada variável a sua contribuição para cada previsão individual, de modo que a soma das contribuições reconstrói a previsão. Para árvores, esses valores são calculados de forma exata.

As contribuições são agrupadas por família de variável, para que a leitura seja feita em termos do problema, e não de colunas isoladas.

In [ ]:
try:
    import shap
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "shap"])
    import shap

AREA_SHAP = "ipc_comp"
X, y, g = DADOS[AREA_SHAP]
modelo = floresta(**ESCOLHA[(AREA_SHAP, "floresta")]).fit(X, y)
Xt = modelo.named_steps["componente"].transform(X)
valores = shap.TreeExplainer(modelo.named_steps["modelo"]).shap_values(Xt)

FAMILIAS = {"patamar": "Patamar do componente", "escala": "Escala do componente",
            "run_case_": "Tipo de ensaio", "losses_": "Perfil de perdas", "reason_": "Motivo do ensaio",
            "blower_direction_": "Direção do soprador", "housing_": "Compartimento",
            "board_position_": "Posição da placa", "deflector_hole_": "Furo do defletor",
            "material_coil_support_": "Suporte das bobinas"}


def familia(coluna):
    for prefixo, nome in FAMILIAS.items():
        if coluna == prefixo or coluna.startswith(prefixo):
            return nome
    return "Outra"


peso = pd.Series(np.abs(valores).mean(axis=0), index=Xt.columns).groupby(familia).sum()
peso = (100 * peso / peso.sum()).sort_values()
print((peso[::-1]).round(1).to_string())

fig, ax = plt.subplots(figsize=(7, 4))
peso.plot.barh(ax=ax, color="#4a6fa5")
ax.set_xlabel("Participação na atribuição média, em %")
ax.set_title(f"Contribuição por família de variável, floresta em {AREA_SHAP}")
plt.tight_layout()
plt.show()

A leitura é coerente com a física da área. O patamar do componente, que informa em que ponto da placa de potência está o sensor, responde por um terço da explicação, o que é esperado: componentes diferentes operam em faixas de temperatura diferentes.

O tipo de ensaio e o perfil de perdas somam cerca de quarenta por cento. Ambos descrevem a carga elétrica imposta, e a placa de potência, que abriga transistores e capacitores, aquece principalmente pela potência que conduz. O motivo do ensaio, com 11,5 por cento, reúne as alterações de projeto que mudam o resfriamento, como a cobertura do dissipador e o caminho do ar.

Os atributos de geometria e montagem, como o furo do defletor, o compartimento, o suporte das bobinas, a direção do soprador e a posição da placa, somam menos de dez por cento. Nenhuma variável fisicamente implausível aparece entre as de maior peso.

Convém registrar o alcance desta leitura. Os valores explicam como o modelo chega às suas previsões, e não demonstram relação de causa no fogão. E o modelo explicado foi ajustado sobre todos os ensaios de desenvolvimento, e não dobra a dobra.

## 9. Do melhor algoritmo ao modelo entregue: perdas por área e peso junto ao limite

A formulação efetivamente entregue no caderno de modelagem parte desta comparação e acrescenta dois passos, ambos medidos naquele caderno.

O primeiro é a codificação do perfil de perdas escolhida por área, entre os treze indicadores detalhados e quatro categorias agrupadas. O segundo é um peso no ajuste que prioriza as leituras próximas do limite de cada componente, porque é ali que a decisão de triagem acontece. Em Main Vent Sys, a floresta dá lugar ao Extra Trees na média com o gradiente, troca identificada na comparação de algoritmos do fim da sprint e adotada apenas nessa região.

A célula abaixo reproduz a formulação entregue com a configuração fixa dos modelos, para comparação direta com as seções anteriores.

In [ ]:
REGRA = {"oven_hmi": ("floresta", "agrupado"), "ct_knobs": ("floresta", "detalhado"),
         "main_vent_sys": ("mescla ET + GB", "detalhado"), "ipc_cooling": ("floresta", "agrupado"),
         "mainback": ("floresta", "detalhado"), "ipc_comp": ("floresta", "agrupado"),
         "coil_temps": ("mescla", "detalhado")}


def agrupa_perdas(X):
    """Niveis exclusivos no lugar dos treze indicadores de perdas.

    Sobre a base atual saem quatro colunas: miro, none, simone e nao declarado. O
    rotulo other existe para o caso de a planilha trazer um perfil que nao seja
    nenhum dos tres, e hoje nao e usado por nenhuma das treze colunas.

    O nivel de nao declarado vem da ausencia de qualquer indicador, e nao de um
    rotulo proprio, porque a planilha deixa a celula vazia nesses ensaios. E a
    mesma construcao de _bloco_agrupado, no caderno de modelagem, inclusive na
    verificacao de exclusividade, que aqui faltava: sem ela, um rotulo novo de
    perdas mudaria o numero de colunas de entrada sem aviso.
    """
    perdas = [c for c in X.columns if c.startswith("losses_")]
    rotulo = lambda c: next((n for k, n in (("no losses", "none"), ("simone", "simone"),
                                              ("miro", "miro")) if k in c.lower()), "other")
    saida = X.drop(columns=perdas).copy()
    ativo = X[perdas].to_numpy(float)
    familias = np.array([rotulo(c) for c in perdas])
    niveis = sorted(set(familias.tolist()))
    for nivel in niveis:
        saida[f"perdas_{nivel}"] = (ativo[:, familias == nivel].sum(axis=1) > 0).astype(float)
    saida["perdas_nao_declarado"] = (ativo.sum(axis=1) == 0).astype(float)
    criadas = [f"perdas_{n}" for n in niveis] + ["perdas_nao_declarado"]
    assert (saida[criadas].sum(axis=1) == 1).all(),         "os niveis de perdas deixaram de ser exclusivos"
    return saida


def peso_limite(y, componentes):
    """Peso tres sobre o limite, caindo para um a cerca de trinta graus dele."""
    lim = componentes.map(LIMITES).to_numpy(dtype=float)
    d = np.nan_to_num(np.abs(y - lim), nan=1e9)
    return np.where(np.isnan(lim), 1.0, 1.0 + 2.0 * np.exp(-(d / 15.0) ** 2))


def prever_entrega(area):
    alg, perdas = REGRA[area]
    X, y, g = DADOS[area]
    Xa = agrupa_perdas(X) if perdas == "agrupado" else X
    pred = np.empty(len(y))
    for treino, teste in divisoes(area, Xa, y, g):
        w = peso_limite(y[treino], X["componente"].iloc[treino])
        # Main Vent Sys usa o Extra Trees no lugar da floresta, na mescla com o gradiente.
        base = extra(n_estimators=300) if alg == "mescla ET + GB" else floresta(n_estimators=300)
        p = base.fit(Xa.iloc[treino], y[treino], modelo__sample_weight=w).predict(Xa.iloc[teste])
        if alg in ("mescla", "mescla ET + GB"):
            gb = gradiente(n_estimators=300).fit(Xa.iloc[treino], y[treino], modelo__sample_weight=w)
            p = (p + gb.predict(Xa.iloc[teste])) / 2
        pred[teste] = p
    return pred


for area in AREAS:
    PREVISAO[(area, "entrega")] = prever_entrega(area)

comparacao = {"Melhor por área": TAB_AGREGADO.loc["Melhor por área"].to_dict(),
              "Formulação entregue": agrega({a: "entrega" for a in AREAS})}
TAB_ENTREGA = pd.DataFrame(comparacao).T
print(TAB_ENTREGA.round(3).to_string())

A formulação entregue erra 5,855 graus, com viés de +0,152, vinte e três aprovações indevidas, cento e quarenta e três reprovações desnecessárias e detecção de 88,1 por cento. Esses valores são os mesmos da primeira versão do caderno, que reproduz a mesma formulação, já com o Extra Trees na mescla de Main Vent Sys. O termo de comparação, a linha do melhor modelo por área, é lido da célula acima e não transcrito aqui, porque agora inclui o Extra Trees entre os candidatos.

| Indicador | Formulação entregue |
|---|---:|
| Erro médio absoluto | 5,855 |
| Viés | +0,152 |
| Aprovações indevidas | 23 |
| Reprovações desnecessárias | 143 |
| Detecção | 88,1% |

O melhor modelo por área deste caderno mede 5,950 graus, com viés de +0,463, vinte e cinco aprovações indevidas e cento e oitenta e duas reprovações desnecessárias. O resultado merece atenção porque a formulação entregue usa os hiperparâmetros fixos, sem busca, e ainda assim fica à frente. A vantagem vem dos dois passos descritos no início desta seção: a codificação das perdas escolhida por área reduz o erro nas três áreas em que o perfil de perdas carrega pouco sinal, e o peso próximo ao limite concentra o ajuste onde a decisão de triagem acontece, o que se traduz em trinta e nove reprovações desnecessárias a menos e num viés três vezes menor.

Dois cuidados devem acompanhar esta comparação. O valor de 5,855 corresponde à regra por área escrita como constante, que foi derivada olhando os resultados. Com a escolha refeita dentro de cada dobra, o mesmo modelo mede entre 5,96 e 6,23 graus, conforme o método de aninhamento, e é essa faixa que o projeto reporta. O limite inferior vem do método usado ao longo da sprint, levemente otimista; o superior, de um método estrito, levemente pessimista. E o valor do melhor por área também é otimista, pela mesma razão, somada ao otimismo da busca de hiperparâmetros já discutido.

A leitura conjunta das seções 7 e 9 sugeria um próximo passo: combinar o gradiente ajustado com o peso e com a codificação das perdas por área. A hipótese foi testada no caderno de modelagem, com a escolha feita de forma aninhada, e não se sustentou. Pelo método de aninhamento usado ao longo da sprint, o ganho parecia ser de 0,107 grau; pelo método estrito, converteu-se numa perda de 0,448 grau. O gradiente ajustado continua melhor em Mainback e em Coil Temps, mas a validação não consegue identificar com segurança onde ele ajuda, e por isso a formulação entregue manteve o gradiente com os valores padrão.

## 10. Ressalvas

A melhor configuração da busca é escolhida e avaliada pela mesma validação, e por isso o seu erro é otimista. Este efeito foi medido com uma validação aninhada, e está registrado em `documents/extras/relatorios_sprint_4/relatorioS4.md`, em que a configuração é escolhida em cada dobra pelo erro nas demais e só então aplicada à dobra retida: o ganho aparente da busca era de 0,506 grau e o ganho aninhado foi de 0,051 grau, com intervalo de confiança que contém o zero (CAWLEY; TALBOT, 2010). Os ganhos da seção 5 devem ser lidos com essa proporção em mente. O mesmo efeito reapareceu, maior, ao testar o gradiente ajustado dentro da formulação entregue: a diferença entre o método de aninhamento aproximado e o estrito chegou a 0,555 grau.

Todos os números vêm de uma semente aleatória. A oscilação entre sementes foi medida nesta base e fica em centésimos de grau.

O conjunto de desenvolvimento tem quarenta e seis ensaios em dezessete configurações distintas, o que limita todas as conclusões deste caderno.

## 11. Quadro consolidado

Esta seção reúne, numa tabela só, o melhor resultado de cada área térmica entre os treinos feitos nos cadernos de `colabs_S4`, com as duas métricas e com o tratamento de dados que aquele resultado utiliza.

Três áreas trazem o resultado treinado neste caderno. Quatro trazem o de outro caderno de `colabs_S4`, porque ali o erro obtido foi menor: Oven HMI, Main Vent Sys e IPC Comp vêm da formulação entregue de `modelagem.ipynb`, Seção 23, e IPC Cooling vem de `random_search_13cat.ipynb`. Esses quatro valores estão escritos como constantes, e não recalculados, porque foram medidos com outra receita ou com uma busca maior. Todas as linhas foram medidas com o corte por configuração.

As quatro colunas de sim ou não registram o tratamento de dados. As duas primeiras dizem se o resultado usa as perdas agrupadas ou as treze categorias detalhadas. As duas últimas dizem se ele usa o motivo do ensaio e, em caso afirmativo, se na forma agrupada em seis níveis mutuamente exclusivos ou na forma anterior, de seis indicadores sobrepostos com prefixo `cfg_`.

### A regra de escolha

Cada área traz, entre os resultados que os cadernos de treino de `colabs_S4` reportam, o de menor erro médio absoluto que atende a três condições. A primeira é não imputar valores ausentes: as leituras sem medição são descartadas, e não preenchidas por média ou mediana, que é a decisão adotada pelo projeto. A segunda é ter sido medido sob a validação que retira uma configuração experimental inteira por vez. A terceira é usar a base corrigida, em que o Oven HMI tem três componentes; a base anterior tinha quatro, dois deles cópias exatas um do outro, e o erro do componente copiado entrava duas vezes na média.

Entram os resultados que cada caderno adota como seus: na modelagem, a formulação entregue, com a escolha do candidato refeita dentro de cada dobra; nos cadernos de busca, o primeiro colocado de cada algoritmo pelo critério da própria busca. Candidatos intermediários ficam de fora, como o candidato isolado da floresta com perdas agrupadas em IPC Comp, na Seção 23 da modelagem, com 6,606, que é medido sem a etapa de escolha.

Também fica de fora a execução anterior do `random_search_13cat`, que descartava as colunas `reason_` por um prefixo não declarado entre as entradas. Nela, o Extra Trees de Main Vent Sys media 3,77 graus, com `n_estimators` 382, `max_depth` None, `min_samples_leaf` 1, `min_samples_split` 10 e `max_features` 1,0. O próprio caderno foi corrigido e reexecutado com as colunas `reason_` e declarou que aqueles números não devem mais ser usados. Na nova execução, a busca escolhe para o Extra Trees os mesmos hiperparâmetros, e com o motivo do ensaio entre as entradas ele mede 3,89 naquela área.

Os cadernos de mesclagem de `testes_S4` ficam de fora por dois motivos: estão fora de `colabs_S4` e treinam sobre os arquivos `_snull`, em que a própria temperatura foi preenchida pelo pré-processamento.

Levantamento dos resultados reportados, com o menor de cada área em negrito. Os valores do `random_search_13cat` são os da execução com as colunas `reason_`, com as duas casas que o caderno imprime; o de IPC Cooling e os dois empates têm a terceira casa, obtida refazendo a busca com a mesma semente.

| Área | este caderno | `modelagem`, Seção 23 | `random_search_13cat` | `random_search` |
|---|---|---|---|---|
| Oven HMI | Floresta 2,393 | **Floresta 2,321** | Gradiente 2,39 | Gradiente 2,318, base antiga, excluído |
| CT Knobs | **Gradiente 2,769** | Floresta 3,051 | **Gradiente 2,769, o mesmo modelo** | Gradiente 3,116 |
| Main Vent Sys | Gradiente 3,868 | **Extra Trees e gradiente 3,739** | Gradiente 3,87; Extra Trees 3,89 | Gradiente 4,663 |
| IPC Cooling | Mescla 3,822 | Floresta 4,066 | **Gradiente 3,710** | Extra Trees 4,144 |
| Mainback | **Gradiente 4,129** | Floresta 4,586 | Gradiente 4,23 | Gradiente 5,390 |
| IPC Comp | Gradiente 6,921 | **Floresta 6,677** | Gradiente 6,75 | Gradiente 7,121 |
| Coil Temps | **Gradiente 10,198** | Floresta e gradiente 10,612 | **Gradiente 10,198, o mesmo modelo** | Gradiente 10,408 |

Em CT Knobs e Coil Temps, este caderno e o `random_search_13cat` chegam ao mesmo gradiente, com os mesmos hiperparâmetros, porque a mesma semente sorteia os mesmos primeiros candidatos nas duas buscas; o quadro traz a linha deste caderno. Em IPC Cooling, a busca do `random_search_13cat`, com quarenta candidatos, encontrou um gradiente que as quinze deste caderno não sortearam: `learning_rate` 0,1318, `loss` absolute_error, `max_depth` 4, `max_features` None, `min_samples_leaf` 14, `n_estimators` 776 e `subsample` 0,8462. Como aquele caderno grava os hiperparâmetros só em arquivos fora do repositório, eles foram obtidos refazendo a busca com a mesma semente, que reproduziu os valores impressos pelo caderno.

Em Main Vent Sys, o resultado é o da formulação entregue: a média entre o Extra Trees e o gradiente, ambos com trezentas árvores e os demais hiperparâmetros no padrão da biblioteca, com as perdas detalhadas e o peso junto ao limite. A escolha aninhada da modelagem elegeu esse candidato em todas as cinquenta e uma dobras e sementes.

Os resultados da modelagem são a média de três sementes, com a escolha do candidato refeita dentro de cada dobra, e por isso são os menos otimistas da tabela; os deste caderno e dos cadernos de busca usam uma semente só e escolhem os hiperparâmetros na mesma validação que os avalia.

Três ressalvas acompanham a tabela e não devem ser separadas dela.

As receitas não são as mesmas em todas as linhas. Oven HMI, Main Vent Sys e IPC Comp vêm da formulação entregue, com hiperparâmetros fixos e o peso junto ao limite do componente, e com as perdas agrupadas em Oven HMI e IPC Comp e detalhadas em Main Vent Sys; IPC Cooling vem de uma busca de quarenta candidatos; as demais, de uma busca de quinze. Todas usam o motivo do ensaio nos seis níveis agrupados. As colunas de sim ou não registram o tratamento das perdas e do motivo; o peso e a busca ficam registrados aqui.

Os coeficientes de determinação da tabela são todos da área inteira, com as previsões agrupadas, que é a definição usada por este caderno, pela modelagem e pela coluna `R2_area_rs` do `random_search_13cat`. Esse caderno imprime também uma média do coeficiente calculado componente a componente, bem mais severa, e as duas não devem ser misturadas.

Todos os valores vêm de validação cruzada com uma configuração experimental retirada por vez, e nenhum deles usa o conjunto de teste reservado.

In [ ]:
# Resultados importados de outros cadernos. Nao sao recalculados aqui, porque foram
# medidos com outra receita ou outra preparacao da base. A origem de cada um fica
# registrada na propria tabela, e a nota descreve a diferenca.
EXTERNOS = {
    "oven_hmi": {
        "melhor_algoritmo": "Floresta, formulação entregue", "MAE": 2.321, "R2": 0.032,
        "losses_agrupadas": "sim", "losses_13_categorias": "nao",
        "usa_reason": "sim", "reason_agrupada": "sim",
        "origem": "modelagem.ipynb, Seção 23",
        "nota": ("floresta de 300 arvores, perdas agrupadas em quatro niveis e peso junto ao "
                 "limite; media de tres sementes com a escolha do candidato refeita dentro de "
                 "cada dobra; base corrigida, com tres componentes e 135 leituras"),
    },
    "ipc_comp": {
        "melhor_algoritmo": "Floresta, formulação entregue", "MAE": 6.677, "R2": 0.662,
        "losses_agrupadas": "sim", "losses_13_categorias": "nao",
        "usa_reason": "sim", "reason_agrupada": "sim",
        "origem": "modelagem.ipynb, Seção 23",
        "nota": ("floresta de 300 arvores, perdas agrupadas em quatro niveis e peso junto ao "
                 "limite; media de tres sementes com a escolha do candidato refeita dentro de "
                 "cada dobra; base corrigida, com 37 componentes e 1616 leituras"),
    },
    "main_vent_sys": {
        "melhor_algoritmo": "Mescla ET + GB, formulação entregue", "MAE": 3.739, "R2": 0.910,
        "losses_agrupadas": "nao", "losses_13_categorias": "sim",
        "usa_reason": "sim", "reason_agrupada": "sim",
        "origem": "modelagem.ipynb, Seção 23",
        "nota": ("media entre ExtraTreesRegressor e GradientBoostingRegressor, ambos com "
                 "n_estimators=300 e os demais hiperparametros no padrao da biblioteca; perdas "
                 "detalhadas e peso junto ao limite; media de tres sementes, com o candidato "
                 "escolhido em 51 de 51 dobras; 230 leituras"),
    },
    "ipc_cooling": {
        "melhor_algoritmo": "Gradiente, busca", "MAE": 3.710, "R2": 0.849,
        "losses_agrupadas": "nao", "losses_13_categorias": "sim",
        "usa_reason": "sim", "reason_agrupada": "sim",
        "origem": "random_search_13cat.ipynb",
        "nota": ("primeiro colocado de quarenta candidatos, execucao com as colunas reason_; "
                 "learning_rate=0.1318, loss=absolute_error, max_depth=4, max_features=None, "
                 "min_samples_leaf=14, n_estimators=776, subsample=0.8462; 179 leituras"),
    },
}

# Tratamento de dados deste caderno: perdas em treze categorias detalhadas e motivo do
# ensaio em seis niveis mutuamente exclusivos, montados na preparacao dos conjuntos.
DESTE_CADERNO = {"losses_agrupadas": "nao", "losses_13_categorias": "sim",
                 "usa_reason": "sim", "reason_agrupada": "sim",
                 "origem": "este caderno", "nota": ""}

resumo = []
for a in AREAS:
    alg_proprio = mae.loc[a, "vencedor"]
    mae_proprio = float(mae.loc[a, alg_proprio])
    r2_proprio = float(r2.loc[a, alg_proprio])
    if a in EXTERNOS:
        linha = {"area": ROTULO[a], **EXTERNOS[a]}
        # Se esta execucao superar o valor importado, o aviso aparece em vez de a
        # tabela relatar em silencio um numero pior do que o proprio caderno mediu.
        if mae_proprio < linha["MAE"]:
            print(f"Aviso: em {ROTULO[a]} este caderno mediu {mae_proprio:.3f} com "
                  f"{alg_proprio}, abaixo dos {linha['MAE']:.3f} importados de "
                  f"{linha['origem']}. Reveja qual dos dois a tabela deve trazer.")
    else:
        linha = {"area": ROTULO[a], "melhor_algoritmo": alg_proprio,
                 "MAE": mae_proprio, "R2": r2_proprio, **DESTE_CADERNO}
    resumo.append(linha)

COLUNAS = ["area", "melhor_algoritmo", "MAE", "R2", "losses_agrupadas",
           "losses_13_categorias", "usa_reason", "reason_agrupada", "origem"]
RESUMO = pd.DataFrame(resumo)[COLUNAS]

print("Melhor resultado conhecido de cada area termica:")
print()
print(RESUMO.round(3).to_string(index=False))

print()
print("Notas de preparacao das areas importadas:")
for l in resumo:
    if l.get("nota"):
        print(f"  {l['area']}: {l['nota']}")

# Mesma tabela em markdown, para a documentacao do artefato.
print()
print("| Área | Melhor algoritmo | MAE | R² | Losses agrupadas | Losses 13 categorias "
      "| Usa reason | Reason agrupada | Origem |")
print("|---|---|---:|---:|:---:|:---:|:---:|:---:|---|")
for _, l in RESUMO.iterrows():
    print(f"| {l['area']} | {l['melhor_algoritmo']} | {l['MAE']:.3f} | {l['R2']:.3f} "
          f"| {l['losses_agrupadas']} | {l['losses_13_categorias']} "
          f"| {l['usa_reason']} | {l['reason_agrupada']} | {l['origem']} |")

## 12. Resumo

As duas tabelas abaixo resumem o caderno, uma linha por área. Os valores da quarta sprint são os do quadro consolidado da Seção 11, e por isso são gerados pela célula, e não transcritos.

A primeira compara o erro médio absoluto e o coeficiente de determinação com os da solução entregue na terceira sprint, registrados na Seção 4.3 da documentação. Os dois foram medidos com a mesma validação, que retira uma configuração experimental inteira por vez. A base da terceira sprint ainda tinha o componente duplicado do Oven HMI, de modo que a linha daquela área compara bases ligeiramente diferentes.

A segunda registra como os dados de entrada do melhor modelo de cada área foram tratados: se as perdas entram agrupadas ou nas treze categorias, e se o motivo do ensaio é aplicado.

In [ ]:
# Solucao entregue na terceira sprint, validacao por configuracao (Secao 4.3 da
# documentacao): erro medio absoluto e coeficiente de determinacao por area.
SPRINT3 = {"oven_hmi": (2.42, -0.16), "ct_knobs": (3.16, 0.56), "main_vent_sys": (4.68, 0.89),
           "ipc_cooling": (4.04, 0.85), "mainback": (5.68, 0.84), "ipc_comp": (7.30, 0.62),
           "coil_temps": (11.71, 0.85)}

S4 = RESUMO.set_index("area")


def algoritmo(a):
    """Nome do algoritmo vencedor, sem o complemento sobre busca ou formulacao."""
    return S4.loc[ROTULO[a], "melhor_algoritmo"].split(",")[0]


RESUMO_METRICAS = pd.DataFrame([
    {"Área": ROTULO[a], "Melhor algoritmo": algoritmo(a),
     "MAE S3": SPRINT3[a][0], "MAE S4": S4.loc[ROTULO[a], "MAE"],
     "R² S3": SPRINT3[a][1], "R² S4": S4.loc[ROTULO[a], "R2"]}
    for a in AREAS])

RESUMO_DADOS = pd.DataFrame([
    {"Área": ROTULO[a], "Melhor algoritmo": algoritmo(a),
     "Losses": "agrupadas" if S4.loc[ROTULO[a], "losses_agrupadas"] == "sim" else "13 categorias",
     "Reason": "aplicada" if S4.loc[ROTULO[a], "usa_reason"] == "sim" else "não aplicada"}
    for a in AREAS])

print("Tabela 1. Métricas por área, terceira e quarta sprints")
print(RESUMO_METRICAS.round(3).to_string(index=False))
print()
print("Tabela 2. Tratamento dos dados de entrada do melhor modelo de cada área")
print(RESUMO_DADOS.to_string(index=False))


def em_markdown(tabela):
    """A tabela em markdown, com virgula decimal, para a documentacao."""
    virgula = lambda v: f"{v:.2f}".replace(".", ",") if isinstance(v, float) else str(v)
    alinha = ["---:" if tabela[c].dtype.kind == "f" else "---" for c in tabela.columns]
    saida = ["| " + " | ".join(tabela.columns) + " |", "|" + "|".join(alinha) + "|"]
    saida += ["| " + " | ".join(virgula(v) for v in linha) + " |"
              for linha in tabela.itertuples(index=False)]
    return "\n".join(saida)


print()
print(em_markdown(RESUMO_METRICAS))
print()
print(em_markdown(RESUMO_DADOS))

## 13. Gráficos

Dois gráficos resumem o caderno. Ambos são gerados a partir das tabelas e dos modelos das seções anteriores, e não de números transcritos.

### 13.1 Da terceira para a quarta sprint

Cada linha é uma área, identificada pelo melhor modelo da quarta sprint no quadro da Seção 11. O ponto cinza é a terceira sprint e o azul é a quarta. A linha que os liga é verde quando a métrica melhorou, vermelha quando piorou e cinza quando não mudou na segunda casa decimal, que é a precisão dos valores da terceira sprint.

In [ ]:
# Grafico 1: de onde a terceira sprint estava para onde a quarta chegou, em cada area.
# Le RESUMO_METRICAS, da Secao 12, para que o grafico nunca divirja da tabela.
def virgula(v, casas=2):
    return f"{v:.{casas}f}".replace(".", ",").replace("-", "\u2212")


COR_S3, COR_S4 = "#9aa5b1", "#1f5f99"
COR_MELHORA, COR_PIORA, COR_IGUAL = "#2e8b57", "#c0392b", "#9aa5b1"
PAINEIS = [("MAE S3", "MAE S4", "Erro médio absoluto, em °C\n(menor é melhor)", True, (0, 14.2)),
           ("R² S3", "R² S4", "Coeficiente de determinação\n(maior é melhor)", False, (-0.3, 1.22))]

ordem = RESUMO_METRICAS.iloc[::-1].reset_index(drop=True)   # a primeira area fica no topo
fig, eixos = plt.subplots(1, 2, figsize=(13, 5.6), sharey=True,
                          gridspec_kw={"width_ratios": [1.3, 1]})
for ax, (s3, s4, titulo, menor_melhor, limites) in zip(eixos, PAINEIS):
    for i, l in ordem.iterrows():
        a, b = float(l[s3]), float(l[s4])
        delta = round(b - a, 2)          # os valores da terceira sprint tem duas casas
        if delta == 0:
            cor = COR_IGUAL
        else:
            cor = COR_MELHORA if (delta < 0) == menor_melhor else COR_PIORA
        ax.plot([a, b], [i, i], color=cor, lw=3, zorder=1, solid_capstyle="round")
        ax.scatter(a, i, s=70, color=COR_S3, zorder=2)
        ax.scatter(b, i, s=95, color=COR_S4, zorder=3)
        sinal = "+" if delta > 0 else ("\u2212" if delta < 0 else "\u00b1")
        ax.annotate(f"{virgula(b)}  ({sinal}{virgula(abs(delta))})", (max(a, b), i),
                    xytext=(10, 0), textcoords="offset points", va="center", fontsize=9.5,
                    color=cor if delta else "#555555")
    ax.set_xlim(*limites)
    ax.set_title(titulo, fontsize=11)
    ax.grid(axis="x", color="#e5e5e5", zorder=0)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(axis="y", length=0)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: virgula(v, 1)))
eixos[0].set_yticks(range(len(ordem)))
eixos[0].set_yticklabels([f"{l['Área']}\n{l['Melhor algoritmo']}" for _, l in ordem.iterrows()],
                         fontsize=10)
eixos[0].scatter([], [], s=70, color=COR_S3, label="Sprint 3")
eixos[0].scatter([], [], s=95, color=COR_S4, label="Sprint 4")
eixos[0].legend(loc="upper right", frameon=False)
fig.suptitle("Melhor modelo de cada área: Sprint 3 e Sprint 4", fontsize=13, x=0.08, ha="left")
fig.text(0.08, -0.02, "Validação por configuração nas duas sprints; na Sprint 3, o Oven HMI ainda tinha "
         "o componente duplicado. Entre parênteses, a variação em relação à Sprint 3.",
         fontsize=8.5, color="#555555")
plt.tight_layout()
plt.show()

O erro médio absoluto cai nas sete áreas. As maiores quedas estão em Mainback, de 1,55 grau, Coil Temps, de 1,51, e Main Vent Sys, de 0,94. O coeficiente de determinação sobe em cinco áreas, fica igual em IPC Cooling e cai dois centésimos em CT Knobs. No Oven HMI, ele passa de negativo para perto de zero, pelos motivos descritos na Seção 7.1.

### 13.2 Beeswarm SHAP de IPC Comp

O gráfico explica o modelo que o quadro da Seção 11 traz para IPC Comp, a área que concentra metade das leituras do projeto. É a formulação entregue: floresta de trezentas árvores, perdas agrupadas e peso junto ao limite. Ele difere da floresta da Seção 8, que usa as perdas detalhadas e não usa o peso.

Cada ponto é uma leitura. A posição horizontal é quanto aquela coluna somou ou subtraiu da temperatura prevista, em graus, a partir de uma previsão de partida igual à média ponderada do treino. A cor é o valor da coluna naquela leitura; nas colunas de sim ou não, vermelho indica que ela está ativa. As colunas estão ordenadas pela contribuição média absoluta, e só as doze primeiras aparecem.

In [ ]:
# Grafico 2: beeswarm SHAP do modelo que o quadro da Secao 11 traz para IPC Comp, a
# formulacao entregue: floresta de 300 arvores, perdas agrupadas e peso junto ao limite.
# Usa agrupa_perdas e peso_limite, da Secao 9, e a semente da entrega do caderno de
# modelagem. A floresta da Secao 8, ao contrario, e a versao com perdas detalhadas e sem peso.
AREA_BEE = "ipc_comp"
assert REGRA[AREA_BEE] == ("floresta", "agrupado"), "a formulacao entregue de IPC Comp mudou"
X_b, y_b, _ = DADOS[AREA_BEE]
Xa_b = agrupa_perdas(X_b)
modelo_b = floresta(n_estimators=300).fit(
    Xa_b, y_b, modelo__sample_weight=peso_limite(y_b, X_b["componente"]))
Xt_b = modelo_b.named_steps["componente"].transform(Xa_b)
explicador_b = shap.TreeExplainer(modelo_b.named_steps["modelo"])
shap_b = explicador_b.shap_values(Xt_b)

base_b = float(np.ravel(explicador_b.expected_value)[0])
print(f"{AREA_BEE}: {len(y_b)} leituras | {Xt_b.shape[1]} colunas de entrada | "
      f"previsao de partida (media ponderada do treino): {base_b:.1f} °C")
media_abs = pd.Series(np.abs(shap_b).mean(axis=0), index=Xt_b.columns).sort_values(ascending=False)
print("Contribuicao media absoluta, em °C, das dez colunas de maior peso:")
print(media_abs.head(10).round(2).to_string())

shap.summary_plot(shap_b, Xt_b, max_display=12, show=False, plot_size=(10, 6.5))
fig = plt.gcf()
ax = fig.axes[0]
ax.set_xlabel("Contribuição para a temperatura prevista, em °C")
ax.set_title("IPC Comp, formulação entregue: contribuição de cada coluna em cada leitura",
             fontsize=12, loc="left")
barra = fig.axes[-1]
barra.set_ylabel("Valor da coluna", fontsize=10)
barra.set_yticklabels(["Baixo", "Alto"])
plt.tight_layout()
plt.show()

O patamar do componente é a coluna de maior peso: componentes de patamar baixo recebem até vinte e cinco graus a menos. Em seguida vem o tipo de ensaio. Nos ensaios `run_case_broil qualitative`, a previsão sobe entre três e doze graus; nos demais, desce na mesma faixa.

O perfil de perdas age no sentido físico esperado. `perdas_simone` eleva a previsão em até oito graus, e `perdas_none`, que marca os ensaios sem perdas impostas, a reduz em até cinco, porque a placa dissipa menos potência. `reason_position_study` eleva a previsão entre dois e nove graus; essa coluna está ativa em três ensaios de broil qualitative, os ensaios 14, 16 e 17, com médias entre 100 e 103 graus.

`reason_not_declared` requer cuidado. Em IPC Comp, ela está ativa num único ensaio de desenvolvimento, o ensaio 0, do tipo pyro, com média de 101,7 graus, contra 82,6 da área. No modelo ajustado sobre todos os ensaios, ela funciona como marcador desse ensaio, e por isso soma entre doze e vinte e seis graus às leituras dele. Na validação por configuração, a coluna nunca está ativa no treino quando o ensaio 0 é previsto, de modo que essa contribuição não se transfere para um ensaio novo. É a mesma limitação descrita na Seção 23 do caderno de modelagem para as colunas ativas num único ensaio, e o gráfico não deve ser lido como efeito de não declarar o motivo.

Como na Seção 8, os valores explicam como o modelo chega às suas previsões, e não demonstram relação de causa no fogão.

## Referências

BERGSTRA, James; BENGIO, Yoshua. Random search for hyper-parameter optimization. Journal of Machine Learning Research, v. 13, p. 281-305, 2012.

BREIMAN, Leo. Random forests. Machine Learning, v. 45, n. 1, p. 5-32, 2001.

CAWLEY, Gavin C.; TALBOT, Nicola L. C. On over-fitting in model selection and subsequent selection bias in performance evaluation. Journal of Machine Learning Research, v. 11, p. 2079-2107, 2010.

FRIEDMAN, Jerome H. Greedy function approximation: a gradient boosting machine. The Annals of Statistics, v. 29, n. 5, p. 1189-1232, 2001.

GEURTS, Pierre; ERNST, Damien; WEHENKEL, Louis. Extremely randomized trees. Machine Learning, v. 63, n. 1, p. 3-42, 2006.

LUNDBERG, Scott M.; LEE, Su-In. A unified approach to interpreting model predictions. In: ADVANCES IN NEURAL INFORMATION PROCESSING SYSTEMS, 30., 2017, Long Beach. Proceedings [...]. Red Hook: Curran Associates, 2017. p. 4765-4774.

PEDREGOSA, Fabian et al. Scikit-learn: machine learning in Python. Journal of Machine Learning Research, v. 12, p. 2825-2830, 2011.